# OpenAIRE Marine Science benchmark

Benchmark Data Gatherer against the curated OpenAIRE scholarly graph of the European Marine Science
community — *A Novel Curated Scholarly Graph Connecting Textual and Data Publications* (JDIQ 2023,
[10.1145/3597310](https://doi.org/10.1145/3597310)), data on [Zenodo 7464120](https://zenodo.org/record/7464120) (CC-BY 4.0).

| graph | content |
|---|---|
| `curated_MES` | 4,047 pubs · 5,488 datasets · 22 software · pub→data edges with `semantics` and curation `status` (`Validated` = OpenAIRE link confirmed, `Added` = found by curators in the PDF) |
| `curated_MES_with_removed_edges` | same + `Removed` edges (OpenAIRE links rejected by curators → curated negatives) |
| `original_MES_community` | uncurated OpenAIRE snapshot |

1,383 edges also carry `mention_position` (DAS, acknowledgments, table caption, in-text pointer) and `citation_type` (formal / informal).

**Workflow** (this directory is the experiment root):
1. `data/` – raw Zenodo dump (git-ignored)
2. `gt/` – ground truth: positives, curated negatives, dataset hierarchy, AutoDDG pairs
3. `input/` – `pmcid` CSVs: full pilot (papers resolvable to PMC), 50-article dev set, held-out test set
4. `output/<config>/dataset_citations.csv` – Data Gatherer runs
5. `eval/` – metrics, FP/FN dumps, recall breakdowns

**Where things run:** prompt tuning runs **locally on the dev set only** (§6a). The full pilot (~850 articles) is too
big to run locally — it runs on the HPC cluster (§6b) once tuning is done and the Data Gatherer image is rebuilt.
Final scores are reported on the test set.

## 0 · Setup

In [1]:
import json
import logging
import re
import sys
import time
import zipfile
from collections import Counter
from pathlib import Path
from urllib.parse import unquote

import pandas as pd
import requests


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError(f"data-gatherer repo root not found above {start}")


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
ROOT = REPO_ROOT / "scripts" / "OpenAIRE"
DATA_DIR, GT_DIR, INPUT_DIR, OUTPUT_DIR, EVAL_DIR = (ROOT / d for d in ("data", "gt", "input", "output", "eval"))
for d in (DATA_DIR, GT_DIR, INPUT_DIR, OUTPUT_DIR, EVAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))
from scripts.experiment_utils import evaluate_performance

ZENODO_URL = "https://zenodo.org/records/7464120/files/curated_MES_files.zip?download=1"
ZIP_PATH = DATA_DIR / "curated_MES_files.zip"
GRAPH_DIR = DATA_DIR / "curated_MES_files"

pd.set_option("display.max_colwidth", 120)
REPO_ROOT, ROOT

(PosixPath('/Users/pietro/Desktop/VIDA-NYU/data-gatherer'),
 PosixPath('/Users/pietro/Desktop/VIDA-NYU/data-gatherer/scripts/OpenAIRE'))

## 1 · Download the Zenodo dump

In [2]:
if not GRAPH_DIR.exists():
    if not ZIP_PATH.exists():
        with requests.get(ZENODO_URL, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(ZIP_PATH, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

sorted(p.relative_to(DATA_DIR) for p in GRAPH_DIR.glob("*/jsonl_files/*.jsonl"))

[PosixPath('curated_MES_files/curated_MES/jsonl_files/authors.jsonl'),
 PosixPath('curated_MES_files/curated_MES/jsonl_files/datasets.jsonl'),
 PosixPath('curated_MES_files/curated_MES/jsonl_files/publications.jsonl'),
 PosixPath('curated_MES_files/curated_MES/jsonl_files/relations.jsonl'),
 PosixPath('curated_MES_files/curated_MES/jsonl_files/software.jsonl'),
 PosixPath('curated_MES_files/curated_MES_with_removed_edges/jsonl_files/authors.jsonl'),
 PosixPath('curated_MES_files/curated_MES_with_removed_edges/jsonl_files/datasets.jsonl'),
 PosixPath('curated_MES_files/curated_MES_with_removed_edges/jsonl_files/publications.jsonl'),
 PosixPath('curated_MES_files/curated_MES_with_removed_edges/jsonl_files/relations.jsonl'),
 PosixPath('curated_MES_files/curated_MES_with_removed_edges/jsonl_files/software.jsonl'),
 PosixPath('curated_MES_files/original_MES_community/jsonl_files/authors.jsonl'),
 PosixPath('curated_MES_files/original_MES_community/jsonl_files/datasets.jsonl'),
 PosixPath('

## 2 · Load graphs

In [3]:
def load_graph(name: str) -> dict:
    d = GRAPH_DIR / name / "jsonl_files"
    return {k: pd.read_json(d / f"{k}.jsonl", lines=True, dtype=False)
            for k in ("publications", "datasets", "software", "relations")}


cur = load_graph("curated_MES")
rem = load_graph("curated_MES_with_removed_edges")

{k: len(v) for k, v in cur.items()}

{'publications': 4047, 'datasets': 5488, 'software': 22, 'relations': 79838}

## 3 · Canonical identifiers

Nodes carry lists of landing-page URLs. We derive:
- **publications** → DOI (most frequent DOI among URLs) and PMCID (from Europe PMC / PMC URLs, then NCBI ID converter on the DOI)
- **datasets / software** → DOI (PANGAEA `10.1594`, figshare `10.6084`, Zenodo `10.5281`, …), falling back to the first URL

In [4]:
DOI_RE = re.compile(r"(10\.\d{4,9}/[^\s?#&]+)", re.I)
PMC_RE = re.compile(r"\b(PMC\d+)\b", re.I)


def _urls(v):
    return v if isinstance(v, list) else ([] if v is None or (isinstance(v, float) and pd.isna(v)) else [v])


def norm_doi(s: str) -> str:
    return unquote(s).strip().lower().rstrip("/.").removesuffix(".pdf")


def best_doi(urls):
    dois = Counter(norm_doi(m.group(1)) for u in _urls(urls) if (m := DOI_RE.search(str(u))))
    return dois.most_common(1)[0][0] if dois else None


def first_pmcid(urls):
    for u in _urls(urls):
        if m := PMC_RE.search(str(u)):
            return m.group(1).upper()
    return None


pubs = cur["publications"].copy()
pubs["title"] = pubs["title"].str[0]
pubs["abstract"] = pubs["description"].map(lambda d: " ".join(d) if isinstance(d, list) else d)
pubs["doi"] = pubs["url"].map(best_doi)
pubs["pmcid"] = pubs["url"].map(first_pmcid)

artifacts = pd.concat([cur["datasets"].assign(artifact_type="dataset"),
                       cur["software"].assign(artifact_type="software")], ignore_index=True)
artifacts["title"] = artifacts["title"].str[0]
artifacts["description_text"] = artifacts["description"].map(lambda d: " ".join(d) if isinstance(d, list) else d)
artifacts["doi"] = artifacts["url"].map(best_doi)
artifacts["identifier"] = artifacts["doi"].fillna(artifacts["url"].map(lambda u: (_urls(u) or [None])[0]))
artifacts["doi_prefix"] = artifacts["doi"].str.extract(r"^(10\.\d+)/", expand=False)

print(f"pubs with DOI: {pubs['doi'].notna().sum()}/{len(pubs)} | PMCID from URLs: {pubs['pmcid'].notna().sum()}")
print(f"artifacts with DOI: {artifacts['doi'].notna().sum()}/{len(artifacts)}")
artifacts["doi_prefix"].value_counts().head(10)

pubs with DOI: 3984/4047 | PMCID from URLs: 829
artifacts with DOI: 5510/5510


doi_prefix
10.1594     3750
10.6084     1282
10.5281      124
10.1371       94
10.5061       73
10.17632      54
10.17882      36
10.5880       11
10.4121        9
10.5517        7
Name: count, dtype: int64

In [5]:
# Resolve remaining PMCIDs through the NCBI ID converter (cached to gt/doi_to_pmcid.json).
# Only DOIs NCBI actually answered for are cached, so rate-limited batches are retried on re-run.
IDCONV_URL = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"


def doi_to_pmcid(dois, batch_size=100, max_retries=5):
    out = {}
    for i in range(0, len(dois), batch_size):
        batch = dois[i:i + batch_size]
        for attempt in range(max_retries):
            r = requests.get(IDCONV_URL, params={"ids": ",".join(batch), "format": "json",
                                                 "tool": "data-gatherer", "email": "data-gatherer@nyu.edu"}, timeout=60)
            if r.status_code != 429:
                break
            time.sleep(2 ** attempt)
        if r.status_code != 200:
            print(f"batch {i}: HTTP {r.status_code}, will retry on next run")
            continue
        for rec in r.json().get("records", []):
            if rec.get("doi"):
                out[norm_doi(rec["doi"])] = rec.get("pmcid")
        time.sleep(0.4)
    return out


cache_path = GT_DIR / "doi_to_pmcid.json"
doi2pmc = json.loads(cache_path.read_text()) if cache_path.exists() else {}
todo = sorted(set(pubs.loc[pubs["pmcid"].isna() & pubs["doi"].notna(), "doi"]) - set(doi2pmc))
if todo:
    doi2pmc.update(doi_to_pmcid(todo))
    cache_path.write_text(json.dumps(doi2pmc, indent=1))

print(f"pubs with PMCID: {pubs['pmcid'].notna().sum()}/{len(pubs)}")

pubs with PMCID: 829/4047


## 4 · Ground truth

One row per publication → artifact edge. Relations are stored in either direction
(`pub IsSupplementedBy ds`, `ds IsReferencedBy pub`, …), so we orient them to `pub_id → artifact_id`
and keep the original `semantics` plus `direction`.

`IsPartOf` edges between datasets give the PANGAEA collection hierarchy → `parent_identifier`, used for lenient matching.

In [6]:
rel = cur["relations"]
PUB_IDS = set(pubs["id"])
ART_IDS = set(artifacts["id"])


def pub_artifact_edges(relations: pd.DataFrame) -> pd.DataFrame:
    r = relations[relations["semantics"] != "HasAuthor"].copy()
    fwd = r["source"].isin(PUB_IDS) & r["target"].isin(ART_IDS)
    bwd = r["target"].isin(PUB_IDS) & r["source"].isin(ART_IDS)
    r = r[fwd | bwd].copy()
    r["direction"] = fwd[fwd | bwd].map({True: "pub→artifact", False: "artifact→pub"})
    r["pub_id"] = r["source"].where(r["direction"] == "pub→artifact", r["target"])
    r["artifact_id"] = r["target"].where(r["direction"] == "pub→artifact", r["source"])
    for col in ("mention_position", "mentioned_element"):
        r[col] = r[col].map(lambda v: "|".join(v) if isinstance(v, list) and v else None) if col in r else None
    if "citation_type" not in r:
        r["citation_type"] = None
    return r[["pub_id", "artifact_id", "semantics", "direction", "status",
              "citation_type", "mention_position", "mentioned_element"]]


# Dataset hierarchy: child IsPartOf parent
part_of = rel[(rel["semantics"] == "IsPartOf") & rel["source"].isin(ART_IDS) & rel["target"].isin(ART_IDS)]
id2ident = artifacts.set_index("id")["identifier"]
parent_of = part_of.groupby("source")["target"].first().map(id2ident)


def enrich(edges: pd.DataFrame) -> pd.DataFrame:
    out = (edges
           .merge(pubs[["id", "doi", "pmcid", "title"]].rename(columns={"id": "pub_id", "doi": "pub_doi", "title": "pub_title"}), on="pub_id")
           .merge(artifacts[["id", "artifact_type", "identifier", "doi_prefix", "title"]]
                  .rename(columns={"id": "artifact_id", "title": "artifact_title"}), on="artifact_id"))
    out["parent_identifier"] = out["artifact_id"].map(parent_of)
    return out.drop_duplicates(["pub_id", "artifact_id", "semantics"])


gt_all = enrich(pub_artifact_edges(rel))
neg = enrich(pub_artifact_edges(rem["relations"]).query("status == 'Removed'"))
# A pub/artifact pair can be removed under one semantics and kept under another: keep only true negatives
neg = neg[~neg.set_index(["pub_id", "artifact_id"]).index.isin(gt_all.set_index(["pub_id", "artifact_id"]).index)]

print(f"positive edges: {len(gt_all):,}  ({gt_all['pub_id'].nunique():,} pubs, {gt_all['artifact_id'].nunique():,} artifacts)")
print(f"negative edges: {len(neg):,}  ({neg['pub_id'].nunique():,} pubs)")
gt_all.groupby(["semantics", "status"]).size().unstack(fill_value=0)

positive edges: 9,692  (3,797 pubs, 5,451 artifacts)
negative edges: 420  (352 pubs)


status,Added,Validated
semantics,,
Cites,1014,354
Documents,90,191
IsCitedBy,0,4
IsReferencedBy,1407,1466
IsSupplementedBy,1589,2698
References,396,483


In [7]:
gt_all.to_parquet(GT_DIR / "marine_gt_edges.parquet", index=False)
neg.to_parquet(GT_DIR / "marine_gt_removed_edges.parquet", index=False)
pubs[["id", "doi", "pmcid", "title", "abstract", "url"]].to_parquet(GT_DIR / "marine_publications.parquet", index=False)
artifacts[["id", "artifact_type", "identifier", "doi", "doi_prefix", "title", "description_text", "url"]] \
    .to_parquet(GT_DIR / "marine_artifacts.parquet", index=False)
sorted(p.name for p in GT_DIR.iterdir())

['autoddg_pairs.parquet',
 'doi_to_pmcid.json',
 'marine_artifacts.parquet',
 'marine_gt_edges.parquet',
 'marine_gt_removed_edges.parquet',
 'marine_publications.parquet']

## 5 · Pilot input (PMC subset)

`k8s/k8s_processor.py` takes a CSV with a `pmcid` column. The pilot covers every curated publication that
resolves to a PMCID — including those with **no** linked artifact, which test precision. The full pilot runs
on the cluster only (`k8s/run_loop.sh` reads its input from `k8s/input/`); locally we run the dev subset (§5b).

Data Gatherer is scored on **datasets** only (software edges are excluded from the GT below).

In [8]:
pilot = pubs.loc[pubs["pmcid"].notna(), ["pmcid", "doi", "id"]].drop_duplicates("pmcid")
PILOT_NAME = "article_ids_marine_pmc.csv"
PILOT_CSV = INPUT_DIR / PILOT_NAME
pilot[["pmcid"]].to_csv(PILOT_CSV, index=False)
# k8s/run_loop.sh reads --input from k8s/input/<name>
pilot[["pmcid"]].to_csv(REPO_ROOT / "k8s" / "input" / PILOT_NAME, index=False)

pilot_ids = set(pilot["pmcid"])
has_ds = gt_all[(gt_all["artifact_type"] == "dataset") & gt_all["pmcid"].isin(pilot_ids)]
print(f"pilot articles: {len(pilot):,} | with ≥1 dataset: {has_ds['pmcid'].nunique():,} | dataset edges: {len(has_ds):,}")
has_ds["doi_prefix"].value_counts().head(8)

pilot articles: 828 | with ≥1 dataset: 758 | dataset edges: 2,376


doi_prefix
10.1594     1023
10.6084      844
10.1371      293
10.5061       88
10.17882      58
10.5281       25
10.5524        9
10.5441        8
Name: count, dtype: int64

### 5b · Dev / test split

Prompt tuning uses a 50-article **dev** set only; the remaining pilot articles are the held-out **test** set for final scores.
Dev is stratified by each article's dominant dataset repository (DOI prefix). The split is written once to `input/` and
re-read on later runs, so it stays fixed.

In [9]:
DEV_SIZE, SEED = 50, 42
DEV_CSV = INPUT_DIR / "article_ids_marine_dev.csv"
TEST_CSV = INPUT_DIR / "article_ids_marine_test.csv"

if DEV_CSV.exists():
    dev_ids = set(pd.read_csv(DEV_CSV)["pmcid"])
else:
    ds_edges = gt_all[(gt_all["artifact_type"] == "dataset") & gt_all["pmcid"].isin(pilot_ids)]
    dominant = ds_edges.groupby("pmcid")["doi_prefix"].agg(lambda s: s.mode().iat[0] if s.notna().any() else "other")
    strata = pd.Series({p: dominant.get(p, "no_dataset") for p in sorted(pilot_ids)})
    top = strata.value_counts().index[:4]
    strata = strata.where(strata.isin(top), "other")
    # proportional allocation, largest remainder
    alloc = strata.value_counts(normalize=True) * DEV_SIZE
    n_per = alloc.astype(int)
    n_per[(alloc - n_per).sort_values(ascending=False).index[:DEV_SIZE - n_per.sum()]] += 1
    dev_ids = set(pd.concat([strata[strata == k].sample(n=n, random_state=SEED) for k, n in n_per.items() if n]).index)
    pd.DataFrame({"pmcid": sorted(dev_ids)}).to_csv(DEV_CSV, index=False)
    pd.DataFrame({"pmcid": sorted(pilot_ids - dev_ids)}).to_csv(TEST_CSV, index=False)

test_ids = pilot_ids - dev_ids
N_DEV = len(dev_ids)
dev_edges = gt_all[(gt_all["artifact_type"] == "dataset") & gt_all["pmcid"].isin(dev_ids)]
print(f"dev: {N_DEV} articles, {len(dev_edges)} dataset edges | test: {len(test_ids)} articles")
dev_edges["doi_prefix"].value_counts().head(8)

dev: 50 articles, 158 dataset edges | test: 778 articles


doi_prefix
10.6084     73
10.1594     43
10.1371     32
10.5061      4
10.17882     3
10.1021      2
10.5281      1
Name: count, dtype: int64

## 6 · Run Data Gatherer

Same CLIs as the BioDMS experiments (`scripts/BioDMS/test.ipynb`); every run writes `dataset_citations.csv`
under `output/`, and the eval in §7 picks up whatever exists.

Order: **6a** tune prompts locally on the dev set → rebuild and push the Data Gatherer image with the tuned prompts
→ **6b** full pilot on the cluster → score on the test set (§7, `EVAL_SPLIT = "test"`).

**6a · Local prompt tuning** (Haiku 5.5, dev set, `k8s/k8s_processor.py` on this machine).

**6b · Full pilot on the HPC cluster** (`k8s/run_loop.sh`, `--no-enrich`) — fine-tuned Flan-T5, free, rerun at will.
Configs mirror BioDMS. Each config uses its own `--job-suffix` (`-mc*`) so S3 outputs don't collide with the BioDMS `-tc*` runs.
The pods run the code baked into the image in `k8s/deterministic-batch-job-template.yaml`, so local changes
(prompts included) reach the cluster only after the image is rebuilt and pushed.

**6c · Claude full-document read on the full pilot** — commented out (paid, and too big to run locally).


In [10]:
import math

T5_MODEL = "hf-vida-nyu/flan-t5-base-dataref-info-extract"
# Data Gatherer image for the cluster runs (built from this branch for the marine benchmark)
IMAGE = "pietromarini/data-gatherer-nvidia:20261008"
SEED_ONTOLOGY = "data_gatherer/config/data_repositories_ontology.json"
N_PILOT = len(pilot)
N_GPUS = 4  # slices run in parallel; run_loop.sh times out each iteration at 2h
MAX_PER_SLICE = math.ceil(N_PILOT / N_GPUS)

# config label → dataset_citations.csv, relative to output/
CONFIGS = {
    "T5 · c3 (semantic k=3 + regex)":        "t5_c3/iter1/dataset_citations.csv",
    "T5 · c4 (chunked FDR)":                 "t5_c4/iter1/dataset_citations.csv",
    "Haiku · c4 (FDR)":                      "haiku_c4/dataset_citations.csv",
    "Haiku-5.5 · c4 (FDR)":                  "haiku55_c4/dataset_citations.csv",
}
print(f"{N_PILOT} articles → {N_GPUS} slices × ≤{MAX_PER_SLICE}")

828 articles → 4 slices × ≤207


### 6a · Local prompt tuning (Haiku 5.5, dev set)

Runs on this machine via `k8s/k8s_processor.py` (synchronous API, no batch). Each variant writes to its own
`output/dev_*` directory — the processor skips articles already in an existing `dataset_citations.csv`, so delete
that directory to rerun a variant from scratch.

- Prompts live in `data_gatherer/prompts/prompt_templates/dataset_prompts/`; `CLAUDE_FDR_FewShot` is the BioDMS baseline,
  `CLAUDE_FDR_FewShot_DOI` adds dataset-DOI examples, where-to-look and DOI-vs-article guidance.
- The local runs use the same ontology as the cluster runs (`SEED_ONTOLOGY`), copied to `ontology/` because
  `--ontology-path` expects a directory containing `open_bio_data_repos.json`.
- Fetched articles are cached in `scripts/exp_input/Local_fetched_data.parquet` (the processor's default
  `--backup-file`), so later variants don't refetch.
- Score with §7 after setting `EVAL_SPLIT = "dev"`.

In [11]:
import shutil

ONTOLOGY_DIR = ROOT / "ontology"
ONTOLOGY_DIR.mkdir(exist_ok=True)
shutil.copy(REPO_ROOT / SEED_ONTOLOGY, ONTOLOGY_DIR / "open_bio_data_repos.json")

# dev config label → dataset_citations.csv, relative to output/
DEV_CONFIGS = {
    "dev · Haiku-5.5 FDR · base prompt": "dev_haiku55_fdr_base/dataset_citations.csv",
    "dev · Haiku-5.5 FDR · DOI prompt":  "dev_haiku55_fdr_doi/dataset_citations.csv",
    "dev · gpt-oss-20b FDR · shortDescr":  "dev_gptoss_fdr_shortdescr/dataset_citations.csv",
    "dev · T5 k8s · c3 (semantic k=3 + regex)":   "dev_t5_c3_k8s/iter1/dataset_citations.csv",
}

In [ ]:
# Haiku 5.5 -- FDR, BioDMS prompt (baseline)
!cd {REPO_ROOT} && python k8s/k8s_processor.py --input {DEV_CSV} \
    --output-dir {OUTPUT_DIR / "dev_haiku55_fdr_base"} \
    --model claude-haiku-5-5 --batch-size {N_DEV} \
    --ontology-path {ONTOLOGY_DIR} \
    --full-document-read true --prompt-name CLAUDE_FDR_FewShot \
    --use-batch-api false

In [ ]:
# Haiku 5.5 -- FDR, DOI-aware prompt
!cd {REPO_ROOT} && python k8s/k8s_processor.py --input {DEV_CSV} \
    --output-dir {OUTPUT_DIR / "dev_haiku55_fdr_doi"} \
    --model claude-haiku-5-5 --batch-size {N_DEV} \
    --ontology-path {ONTOLOGY_DIR} \
    --full-document-read true --prompt-name CLAUDE_FDR_FewShot_DOI \
    --use-batch-api false

#### gpt-oss-20b (self-hosted vLLM on the cluster)

The `vllm-gptoss` deployment in the `data-gatherer` namespace serves `openai/gpt-oss-20b` on port 8000; its
NetworkPolicy only admits pods from the same namespace, so from this machine reach it through a port-forward.
`.env` already sets `VLLM_CLIENT=http://localhost:8000/v1`. In a separate terminal, keep this running:

```bash
kubectl port-forward -n data-gatherer svc/vllm-gptoss 8000:8000
```

Check it with `curl http://localhost:8000/v1/models`. gpt-oss uses the existing GPT full-document prompt
`GPT_FDR_FewShot_shortDescr` (`system > user`) with its matching schema, which adds context, keywords and
`citation_type` (Primary/Secondary); there is no GPT version of the DOI prompt.

In [ ]:
# gpt-oss-20b (vLLM) -- FDR, GPT_FDR_FewShot_shortDescr + matching schema (adds context, keywords, citation_type)
!cd {REPO_ROOT} && python3 k8s/k8s_processor.py --input {DEV_CSV} \
    --output-dir {OUTPUT_DIR / "dev_gptoss_fdr_shortdescr"} \
    --model vllm-openai/gpt-oss-20b --batch-size {N_DEV} \
    --ontology-path {ONTOLOGY_DIR} \
    --full-document-read true --intelligent-chunking true --prompt-name GPT_FDR_FewShot_shortDescr \
    --response-format dataset_response_schema_with_use_description_and_short \
    --use-batch-api false

### 6b · Full pilot on the HPC cluster (fine-tuned Flan-T5)

Run after building and pushing `IMAGE` (set in the config cell) from this branch:

```bash
docker build --platform linux/amd64 -t pietromarini/data-gatherer-nvidia:20261008 .
docker push pietromarini/data-gatherer-nvidia:20261008
```

The first cell runs the 50-article dev set on one GPU as a check of the new image; compare it with the local T5 run in §7 (`EVAL_SPLIT = "dev"`) before launching the full pilot.

In [ ]:
# Flan-T5-finetuned -- RS3 (c3) on the 50-article dev set, on the cluster: smoke test of IMAGE before the full pilot
DEV_NAME = DEV_CSV.name
shutil.copy(DEV_CSV, REPO_ROOT / "k8s" / "input" / DEV_NAME)  # run_loop.sh reads --input from k8s/input/
!cd {REPO_ROOT} && bash k8s/run_loop.sh \
    --iterations 1 --gpus 1 \
    --input {DEV_NAME} \
    --max-articles-per-slice {N_DEV} \
    --output-dir {OUTPUT_DIR / "dev_t5_c3_k8s"} \
    --seed-ontology {SEED_ONTOLOGY} \
    --model {T5_MODEL} \
    --image {IMAGE} \
    --job-suffix -mdev3 \
    --semantic-retrieval true \
    --top-k 3 \
    --brute-force-regex true \
    --no-enrich

In [ ]:
# Flan-T5-finetuned -- RS3 (c3)
!cd {REPO_ROOT} && bash k8s/run_loop.sh \
    --iterations 1 --gpus {N_GPUS} \
    --input {PILOT_NAME} \
    --max-articles-per-slice {MAX_PER_SLICE} \
    --output-dir {OUTPUT_DIR / "t5_c3"} \
    --seed-ontology {SEED_ONTOLOGY} \
    --model {T5_MODEL} \
    --image {IMAGE} \
    --job-suffix -mc3 \
    --semantic-retrieval true \
    --top-k 3 \
    --brute-force-regex true \
    --no-enrich

In [ ]:
# Flan-T5-finetuned -- Full Document Chunk (c4)
!cd {REPO_ROOT} && bash k8s/run_loop.sh \
    --iterations 1 --gpus {N_GPUS} \
    --input {PILOT_NAME} \
    --max-articles-per-slice {MAX_PER_SLICE} \
    --output-dir {OUTPUT_DIR / "t5_c4"} \
    --seed-ontology {SEED_ONTOLOGY} \
    --model {T5_MODEL} \
    --image {IMAGE} \
    --job-suffix -mc4 \
    --top-k all \
    --semantic-retrieval false \
    --brute-force-regex false \
    --no-enrich

In [ ]:
# GPU energy per T5 config (sums all slices)
from scripts.experiment_utils import compute_gpu_energy_wh

for cfg in (p.split("/")[0] for p in CONFIGS.values() if p.startswith("t5_")):
    logs = sorted((OUTPUT_DIR / cfg / "iter1").glob("slice_*_gpu_power.csv"))
    if logs:
        print(cfg, [compute_gpu_energy_wh(str(p)) for p in logs])

### 6c · Claude full-document read on the full pilot (paid, commented out)

In [ ]:
# # Claude Haiku 4.5 -- FDR
# !cd {REPO_ROOT} && python k8s/k8s_processor.py --input {PILOT_CSV} \
#     --output-dir {OUTPUT_DIR / "haiku_c4"} \
#     --model claude-haiku-4-5-20251001 --batch-size {N_PILOT} \
#     --full-document-read true --prompt-name CLAUDE_FDR_FewShot \
#     --use-batch-api false

In [ ]:
# # Claude Haiku 5.5 -- FDR (adaptive thinking: no temperature, max_tokens raised to 16k)
# !cd {REPO_ROOT} && python k8s/k8s_processor.py --input {PILOT_CSV} \
#     --output-dir {OUTPUT_DIR / "haiku55_c4"} \
#     --model claude-haiku-5-5 --batch-size {N_PILOT} \
#     --full-document-read true --prompt-name CLAUDE_FDR_FewShot \
#     --use-batch-api false

In [ ]:
# # Collect async batch results — only if the FDR run above is switched to --use-batch-api true
# from data_gatherer.data_gatherer import DataGatherer
# dg = DataGatherer(log_level="INFO")
# batch_id, provider, cfg = "msgbatch_...", "anthropic", "haiku_c4"
# resp = OUTPUT_DIR / cfg / "batch_resp.jsonl"
# dg.parser.llm_client.download_batch_results(batch_id=batch_id, output_file_path=str(resp), api_provider=provider)
# dg.from_batch_resp_file_to_df(str(resp), output_file_path=str(OUTPUT_DIR / cfg / "dataset_citations.csv"))

## 7 · Evaluate

- **Article-level P/R/F1** with the shared `evaluate_performance` (macro-averaged, substring alias matching) — directly comparable to the BioDMS numbers.
- **Edge-level recall** broken down by curation `status` (`Added` = links OpenAIRE missed), `semantics`, repository and `mention_position`.
- **Lenient** recall: a prediction of the PANGAEA parent collection also counts.
- **Removed hits**: predicted links the curators rejected — FPs shared with the uncurated OpenAIRE graph.

In [ ]:
class _Orchestrator:
    """evaluate_performance only needs orchestrator.logger."""
    def __init__(self):
        self.logger = logging.getLogger("eval")


logging.basicConfig(level=logging.WARNING, format="%(message)s")
ORCH = _Orchestrator()

# "dev" scores the local tuning runs (DEV_CONFIGS) on the dev set; "test" scores the full runs (CONFIGS) on held-out articles
EVAL_SPLIT = "dev"
EVAL_IDS, RUN_CONFIGS = (dev_ids, DEV_CONFIGS) if EVAL_SPLIT == "dev" else (test_ids, CONFIGS)

GT = gt_all[(gt_all["artifact_type"] == "dataset") & gt_all["pmcid"].isin(EVAL_IDS)].copy()
GT_EVAL = GT[["pmcid", "identifier"]].drop_duplicates()
GT_BASE = [f"https://www.ncbi.nlm.nih.gov/pmc/articles/{p}" for p in sorted(EVAL_IDS)]
NEG = neg[(neg["artifact_type"] == "dataset") & neg["pmcid"].isin(EVAL_IDS)]


def load_predictions(rel_path: str) -> pd.DataFrame | None:
    path = OUTPUT_DIR / rel_path
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df = df[df["dataset_identifier"].notna()].copy()
    df["pmcid"] = df["source_url"].str.extract(r"(PMC\d+)", flags=re.I, expand=False).str.upper()
    df["pred"] = df["dataset_identifier"].astype(str).str.strip().str.lower().map(norm_doi)
    return df


def _alias(a: str, b) -> bool:
    return isinstance(b, str) and bool(b) and (a in b.lower() or b.lower() in a)


def mark_found(edges: pd.DataFrame, pred: pd.DataFrame, col: str = "identifier") -> pd.Series:
    by_pub = pred.groupby("pmcid")["pred"].apply(set).to_dict()
    return edges.apply(lambda r: any(_alias(p, r[col]) for p in by_pub.get(r["pmcid"], ())), axis=1)

In [ ]:
rows, edge_results = [], {}
for label, rel_path in RUN_CONFIGS.items():
    cfg = rel_path.split("/")[0]
    pred = load_predictions(rel_path)
    if pred is not None:
        pred = pred[pred["pmcid"].isin(EVAL_IDS)]
    if pred is None:
        print(f"[skip] {label}: no output/{rel_path}")
        continue
    m = evaluate_performance(pred, GT_EVAL, ORCH,
                             str(EVAL_DIR / f"fp_{EVAL_SPLIT}_{cfg}.txt"),
                             false_negatives_file=str(EVAL_DIR / f"fn_{EVAL_SPLIT}_{cfg}.txt"),
                             repo_return=True, gt_base=GT_BASE)
    e = GT.copy()
    e["found"] = mark_found(e, pred)
    e["found_lenient"] = e["found"] | mark_found(e, pred, "parent_identifier")
    edge_results[cfg] = e
    removed_hits = mark_found(NEG, pred).sum() if len(NEG) else 0
    rows.append({"config": label, "articles": pred["pmcid"].nunique(), "predictions": len(pred),
                 "P": m["average_precision"], "R": m["average_recall"], "F1": m["f1_score"],
                 "edge_recall": e["found"].mean(), "edge_recall_lenient": e["found_lenient"].mean(),
                 "removed_hits": removed_hits})

summary = pd.DataFrame(rows)
if len(summary):
    summary.to_csv(EVAL_DIR / f"summary_{EVAL_SPLIT}.csv", index=False)
summary.round(3)

In [ ]:
def recall_by(col: str) -> pd.DataFrame:
    return pd.DataFrame({cfg: e.groupby(col, dropna=False)["found"].mean() for cfg, e in edge_results.items()}) \
             .assign(n=GT.groupby(col, dropna=False).size()).sort_values("n", ascending=False)


if edge_results:
    for col in ("status", "semantics", "doi_prefix", "mention_position", "citation_type"):
        display(recall_by(col).head(12).round(3))

## 8 · AutoDDG pairs

Dataset ↔ paper pairs for description-generation experiments. Papers that **supplement** or **document** a dataset
are the most informative sources. ~20% of dataset descriptions are near-copies of a linked paper's abstract —
flagged as `abstract_leak` so they can be excluded or analysed separately.

In [ ]:
from difflib import SequenceMatcher

LEAK_THRESHOLD = 0.8

pairs = (gt_all[gt_all["artifact_type"] == "dataset"]
         [["pub_id", "artifact_id", "semantics", "status", "identifier", "pub_doi", "pmcid"]]
         .merge(pubs[["id", "abstract"]].rename(columns={"id": "pub_id"}), on="pub_id")
         .merge(artifacts[["id", "title", "description_text"]].rename(columns={"id": "artifact_id", "title": "dataset_title"}),
                on="artifact_id"))


def _sim(a, b):
    if not isinstance(a, str) or not isinstance(b, str) or not a or not b:
        return None
    sm = SequenceMatcher(None, a[:1500], b[:1500], autojunk=False)
    # quick_ratio() is an upper bound on ratio(): skip the expensive call when it can't cross the threshold
    return sm.ratio() if sm.quick_ratio() > LEAK_THRESHOLD else sm.quick_ratio()


pairs["abstract_similarity"] = [_sim(a, b) for a, b in zip(pairs["abstract"], pairs["description_text"])]
pairs["abstract_leak"] = pairs["abstract_similarity"] > LEAK_THRESHOLD
pairs.to_parquet(GT_DIR / "autoddg_pairs.parquet", index=False)

print(f"pairs: {len(pairs):,} | datasets: {pairs['artifact_id'].nunique():,} | "
      f"datasets with a leaking abstract: {pairs.loc[pairs['abstract_leak'], 'artifact_id'].nunique():,}")
pairs.groupby("semantics")["abstract_leak"].agg(["size", "mean"]).round(3)